In [176]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import statsmodels.api as sm

In [177]:
df = pd.read_csv("data/1_min_SPY_2008-2021.csv")
cay = pd.read_csv("data/cay_current.csv")
shiller = pd.read_excel("data/ie_data.xls", sheet_name="Data", engine="xlrd", skiprows=7)

In [178]:
shiller

,Date,P,D,E,CPI,Fraction,Rate GS10,Price,Dividend,Price.1,...,CAPE,Unnamed: 13,TR CAPE,Unnamed: 15,Yield,Returns,Returns.1,Real Return,Real Return.1,Returns.2
0,1871.01,4.44,0.26,0.400000,12.464061,1871.041667,5.32,116.818288,6.840711,1.168183e+02,...,NaN,NaN,NaN,NaN,NaN,1.004177,1.000000,0.130609,0.092504,0.038106
1,1871.02,4.5,0.26,0.400000,12.844641,1871.125000,5.323333,114.888874,6.638024,1.154420e+02,...,NaN,NaN,NaN,NaN,NaN,1.004180,0.974424,0.130858,0.094635,0.036224
2,1871.03,4.61,0.26,0.400000,13.034972,1871.208333,5.326667,115.978708,6.541098,1.170848e+02,...,NaN,NaN,NaN,NaN,NaN,1.004183,0.964209,0.130951,0.096186,0.034765
3,1871.04,4.74,0.26,0.400000,12.559226,1871.291667,5.33,123.766437,6.788876,1.255180e+02,...,NaN,NaN,NaN,NaN,NaN,1.004185,1.004919,0.122056,0.090972,0.031084
4,1871.05,4.86,0.26,0.400000,12.273812,1871.375000,5.333333,129.850689,6.946745,1.322754e+02,...,NaN,NaN,NaN,NaN,NaN,1.004188,1.032591,0.122638,0.089488,0.033150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860,2026.01,6929.122,79.61,261.306667,325.252,2026.041667,4.21,6986.274922,80.266641,4.742985e+06,...,39.593576,NaN,42.344446,NaN,0.015354,1.010009,40.585060,NaN,NaN,NaN
1861,2026.02,6893.806316,79.70,269.503333,326.785,2026.125000,4.13,6918.061266,79.980414,4.701199e+06,...,38.942641,NaN,41.630283,NaN,0.016976,0.993745,40.798973,NaN,NaN,NaN
1862,2026.03,6654.419091,79.79,277.700000,327.5515,2026.208333,4.25,6662.205061,79.883358,4.531855e+06,...,37.238865,NaN,39.792985,NaN,0.016749,0.999511,40.448902,NaN,NaN,NaN
1863,2026.04,6575.32,NaN,NaN,327.93475,2026.291667,4.3,6575.320000,NaN,4.472753e+06,...,36.483036,NaN,38.930660,NaN,0.016438,NaN,40.381864,NaN,NaN,NaN


In [179]:
df_5m = (
    df.drop(columns=["Unnamed: 0"], errors="ignore")
      .assign(date=pd.to_datetime(df["date"], errors="coerce"))
      .dropna(subset=["date"])
      .set_index("date")
      .resample("5min")
      .agg({
          "open": "first",
          "high": "max",
          "low": "min",
          "close": "last",
          "volume": "sum",
          "barCount": "sum",
          "average": "mean"
      })
      .dropna(subset=["open", "high", "low", "close"])
      .reset_index()
)

df_5m

,date,open,high,low,close,volume,barCount,average
0,2008-01-22 07:30:00,126.45,127.18,126.00,126.78,83061,12343,126.7136
1,2008-01-22 07:35:00,126.80,127.89,126.12,127.65,69696,11408,127.3064
2,2008-01-22 07:40:00,127.58,128.26,127.00,128.10,78193,12310,127.9490
3,2008-01-22 07:45:00,128.08,129.30,128.03,129.03,90224,13043,128.6154
4,2008-01-22 07:50:00,129.02,129.86,128.92,129.25,113235,14669,129.3644
...,...,...,...,...,...,...,...,...
286554,2021-05-06 13:35:00,417.15,417.70,417.09,417.68,9305,4842,417.4136
286555,2021-05-06 13:40:00,417.68,417.98,417.66,417.79,12423,6660,417.7988
286556,2021-05-06 13:45:00,417.78,418.07,417.65,418.05,14996,6841,417.8272
286557,2021-05-06 13:50:00,418.04,418.84,418.03,418.56,43476,18746,418.3868


In [180]:
df.head(391)

,Unnamed: 0,date,open,high,low,close,volume,barCount,average
0,0,20090522 07:30:00,89.45,89.46,89.37,89.37,7872,2102,89.424
1,1,20090522 07:31:00,89.38,89.53,89.37,89.50,5336,1938,89.468
2,2,20090522 07:32:00,89.51,89.54,89.48,89.49,3349,1184,89.516
3,3,20090522 07:33:00,89.49,89.49,89.31,89.34,3495,1240,89.386
4,4,20090522 07:34:00,89.33,89.46,89.33,89.39,9731,2637,89.379
...,...,...,...,...,...,...,...,...,...
386,386,20090522 13:56:00,89.05,89.08,89.01,89.01,6521,1547,89.050
387,387,20090522 13:57:00,89.01,89.05,89.01,89.03,9434,2082,89.031
388,388,20090522 13:58:00,89.03,89.10,89.02,89.04,9984,2238,89.066
389,389,20090522 13:59:00,89.04,89.12,89.03,89.05,11449,2493,89.076


In [181]:
START = "2008-01-01"
END = "2021-12-31"

# Monatsultimo-Close (nur close) und daraus Monthly Returns berechnen
monthly_close = (
    df_5m.set_index("date")["close"]
        .sort_index()
        .loc[START:END]
        .resample("ME")
        .last()
        .to_frame(name="close")
)

monthly_returns = monthly_close.pct_change().rename(columns={"close": "monthly_return"})

monthly_returns.head(), monthly_returns.tail()

(            monthly_return
 date                      
 2008-01-31             NaN
 2008-02-29       -0.025783
 2008-03-31       -0.013980
 2008-04-30        0.049283
 2008-05-31        0.013585,
             monthly_return
 date                      
 2021-01-31       -0.013195
 2021-02-28        0.029179
 2021-03-31        0.042132
 2021-04-30        0.053123
 2021-05-31        0.004266)

In [182]:
# 1. Log prices
df_5m["log_price"] = np.log(df_5m["close"])

# 2. Returns
df_5m["ret"] = df_5m["log_price"].diff()

# 3. Trading hours filtern (optional aber empfohlen)
df_5m = df_5m.set_index("date")
df_5m = df_5m.sort_index()
df_5m = df_5m.between_time("07:30", "16:00")

# 4. Daily RV
rv_daily = df_5m.groupby(df_5m.index.normalize())["ret"].apply(lambda x: np.sum(x**2))
rv_daily.index.name = "date"

# 5. Monthly RV
rv_monthly = rv_daily.resample("ME").sum().to_frame(name="RV")

In [183]:
rv_monthly["RV"] = np.sqrt(rv_monthly["RV"])

In [184]:
vix = yf.download("^VIX", start=START, end=END)

# Implied Variance
vix["IV"] = (vix["Close"] / 100)**2

# Monthly
iv_monthly = vix["IV"].resample("ME").mean().to_frame()

[*********************100%***********************]  1 of 1 completed


In [185]:
vrp = iv_monthly.join(rv_monthly, how="inner")
vrp["VRP"] = vrp["IV"] - vrp["RV"]

In [186]:
monthly_returns["log_ret"] = np.log(1 + monthly_returns["monthly_return"])

In [187]:
shiller

,Date,P,D,E,CPI,Fraction,Rate GS10,Price,Dividend,Price.1,...,CAPE,Unnamed: 13,TR CAPE,Unnamed: 15,Yield,Returns,Returns.1,Real Return,Real Return.1,Returns.2
0,1871.01,4.44,0.26,0.400000,12.464061,1871.041667,5.32,116.818288,6.840711,1.168183e+02,...,NaN,NaN,NaN,NaN,NaN,1.004177,1.000000,0.130609,0.092504,0.038106
1,1871.02,4.5,0.26,0.400000,12.844641,1871.125000,5.323333,114.888874,6.638024,1.154420e+02,...,NaN,NaN,NaN,NaN,NaN,1.004180,0.974424,0.130858,0.094635,0.036224
2,1871.03,4.61,0.26,0.400000,13.034972,1871.208333,5.326667,115.978708,6.541098,1.170848e+02,...,NaN,NaN,NaN,NaN,NaN,1.004183,0.964209,0.130951,0.096186,0.034765
3,1871.04,4.74,0.26,0.400000,12.559226,1871.291667,5.33,123.766437,6.788876,1.255180e+02,...,NaN,NaN,NaN,NaN,NaN,1.004185,1.004919,0.122056,0.090972,0.031084
4,1871.05,4.86,0.26,0.400000,12.273812,1871.375000,5.333333,129.850689,6.946745,1.322754e+02,...,NaN,NaN,NaN,NaN,NaN,1.004188,1.032591,0.122638,0.089488,0.033150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860,2026.01,6929.122,79.61,261.306667,325.252,2026.041667,4.21,6986.274922,80.266641,4.742985e+06,...,39.593576,NaN,42.344446,NaN,0.015354,1.010009,40.585060,NaN,NaN,NaN
1861,2026.02,6893.806316,79.70,269.503333,326.785,2026.125000,4.13,6918.061266,79.980414,4.701199e+06,...,38.942641,NaN,41.630283,NaN,0.016976,0.993745,40.798973,NaN,NaN,NaN
1862,2026.03,6654.419091,79.79,277.700000,327.5515,2026.208333,4.25,6662.205061,79.883358,4.531855e+06,...,37.238865,NaN,39.792985,NaN,0.016749,0.999511,40.448902,NaN,NaN,NaN
1863,2026.04,6575.32,NaN,NaN,327.93475,2026.291667,4.3,6575.320000,NaN,4.472753e+06,...,36.483036,NaN,38.930660,NaN,0.016438,NaN,40.381864,NaN,NaN,NaN


In [188]:
shiller = shiller[["Date", "P", "D", "E"]].copy()

shiller["Date"] = pd.to_datetime(
    shiller["Date"].map(lambda x: f"{x:.2f}"),
    format="%Y.%m",
    errors="coerce"
)

In [189]:
shiller = shiller.dropna(subset=["Date"])
shiller = shiller.set_index("Date")

In [190]:
p = pd.to_numeric(shiller["P"], errors="coerce")
e = pd.to_numeric(shiller["E"], errors="coerce")
d = pd.to_numeric(shiller["D"], errors="coerce")

shiller["log_PE"] = np.log(p / e)
shiller["log_PD"] = np.log(p / d)
shiller = shiller[["log_PE", "log_PD"]].dropna()

In [191]:
cay["date"] = pd.to_datetime(cay["date"])
cay = cay.set_index("date")

cay_monthly = cay[["cay=c-0.218 a-0.801y+0.441"]].rename(
    columns={"cay=c-0.218 a-0.801y+0.441": "CAY"}
)

In [192]:
START = "2008-01-01"
END = "2021-12-31"

shiller = shiller.sort_index().loc[START:END]
cay = cay.sort_index().loc[START:END]

In [193]:
shiller.index = shiller.index + pd.offsets.MonthEnd(0)

In [194]:
cay_monthly = cay_monthly.resample("ME").ffill()

In [195]:
monthly_returns.index = pd.to_datetime(monthly_returns.index)
vrp.index = pd.to_datetime(vrp.index)
shiller.index = pd.to_datetime(shiller.index)
cay_monthly.index = pd.to_datetime(cay_monthly.index)

In [196]:
start = "2000-01-01"  # früher starten wegen rolling mean!
end = "2021-12-31"

fred_series = {
    "BAA": "BAA",        # Corporate BAA yield
    "AAA": "AAA",        # Corporate AAA yield
    "GS10": "GS10",      # 10Y Treasury
    "TB3MS": "TB3MS"
}

data = {}

for name, code in fred_series.items():
    data[name] = web.DataReader(code, "fred", start, end)

fred = pd.concat(data.values(), axis=1)
fred.columns = fred_series.keys()

fred.head()

,BAA,AAA,GS10,TB3MS
DATE,,,,
2000-01-01,8.33,7.78,6.66,5.32
2000-02-01,8.29,7.68,6.52,5.55
2000-03-01,8.37,7.68,6.26,5.69
2000-04-01,8.40,7.64,5.99,5.66
2000-05-01,8.90,7.99,6.44,5.79


In [197]:
# auf Monatsende resamplen
fred = fred.resample("ME").last()

# Index → Spalte
fred = fred.reset_index().rename(columns={"DATE": "Date"})

In [198]:
fred

,Date,BAA,AAA,GS10,TB3MS
0,2000-01-31,8.33,7.78,6.66,5.32
1,2000-02-29,8.29,7.68,6.52,5.55
2,2000-03-31,8.37,7.68,6.26,5.69
3,2000-04-30,8.40,7.64,5.99,5.66
4,2000-05-31,8.90,7.99,6.44,5.79
...,...,...,...,...,...
259,2021-08-31,3.24,2.55,1.28,0.05
260,2021-09-30,3.23,2.53,1.37,0.04
261,2021-10-31,3.35,2.68,1.58,0.05
262,2021-11-30,3.28,2.62,1.56,0.05


In [199]:
fred["DFSP"] = fred["BAA"] - fred["AAA"]

In [200]:
fred["TMSP"] = fred["GS10"] - fred["TB3MS"]

In [201]:
fred["RREL"] = fred["TB3MS"] - fred["TB3MS"].rolling(12).mean()

In [202]:
fred

,Date,BAA,AAA,GS10,TB3MS,DFSP,TMSP,RREL
0,2000-01-31,8.33,7.78,6.66,5.32,0.55,1.34,NaN
1,2000-02-29,8.29,7.68,6.52,5.55,0.61,0.97,NaN
2,2000-03-31,8.37,7.68,6.26,5.69,0.69,0.57,NaN
3,2000-04-30,8.40,7.64,5.99,5.66,0.76,0.33,NaN
4,2000-05-31,8.90,7.99,6.44,5.79,0.91,0.65,NaN
...,...,...,...,...,...,...,...,...
259,2021-08-31,3.24,2.55,1.28,0.05,0.69,1.23,-1.000000e-02
260,2021-09-30,3.23,2.53,1.37,0.04,0.70,1.33,-1.416667e-02
261,2021-10-31,3.35,2.68,1.58,0.05,0.67,1.53,6.938894e-18
262,2021-11-30,3.28,2.62,1.56,0.05,0.66,1.51,3.333333e-03


In [203]:
fred = fred.copy()

fred["Date"] = pd.to_datetime(fred["Date"])
fred = fred.set_index("Date")

# optional: nur relevante Variablen
fred = fred[["BAA", "AAA", "GS10", "TB3MS", "DFSP", "TMSP", "RREL"]]

In [204]:
data = (
    monthly_returns[["monthly_return", "log_ret"]]
    .join(vrp[["IV", "RV", "VRP"]], how="inner")
    .join(shiller, how="inner")
    .join(cay_monthly, how="left")
    .join(fred, how="left")
)

In [205]:
rf_annual = data["TB3MS"] / 100

# Annual → Monthly (log-kompatibel)
rf_monthly = np.log(1 + rf_annual) / 12

# Excess return
data["excess_ret"] = data["monthly_return"] - rf_monthly

In [206]:
data

,monthly_return,log_ret,IV,RV,VRP,log_PE,log_PD,CAY,BAA,AAA,GS10,TB3MS,DFSP,TMSP,RREL,excess_ret
2008-01-31,NaN,NaN,0.067246,0.061013,0.006234,3.066158,3.899597,0.006081,6.54,5.33,3.74,2.75,1.21,0.99,-1.417500,NaN
2008-02-29,-0.025783,-0.026121,0.065171,0.058995,0.006176,3.079178,3.875335,0.006081,6.82,5.53,3.74,2.12,1.29,1.62,-1.805000,-0.027531
2008-03-31,-0.013980,-0.014079,0.073852,0.075129,-0.001277,3.082243,3.840204,0.009843,6.89,5.51,3.51,1.26,1.38,2.25,-2.358333,-0.015024
2008-04-30,0.049283,0.048108,0.046684,0.046326,0.000358,3.173155,3.875230,0.009843,6.97,5.55,3.68,1.29,1.42,2.39,-2.030000,0.048215
2008-05-31,0.013585,0.013493,0.033596,0.037002,-0.003406,3.250590,3.894051,0.009843,6.93,5.57,3.88,1.73,1.36,2.15,-1.340000,0.012155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-01-31,-0.013195,-0.013283,0.063758,0.040120,0.023637,3.582525,4.179569,NaN,3.24,2.45,1.08,0.08,0.79,1.00,-0.165000,-0.013261
2021-02-28,0.029179,0.028761,0.054342,0.037756,0.016587,3.503641,4.206646,NaN,3.42,2.70,1.26,0.04,0.72,1.22,-0.081667,0.029145
2021-03-31,0.042132,0.041269,0.048388,0.050433,-0.002045,3.417831,4.217321,NaN,3.74,3.04,1.61,0.03,0.70,1.58,-0.070000,0.042107
2021-04-30,0.053123,0.051760,0.030378,0.026464,0.003913,3.398683,4.273294,NaN,3.60,2.90,1.64,0.02,0.70,1.62,-0.070000,0.053106


In [212]:
data.isna().sum()

monthly_return     0
log_ret            0
IV                 0
RV                 0
VRP                0
log_PE             0
log_PD             0
CAY               20
BAA                0
AAA                0
GS10               0
TB3MS              0
DFSP               0
TMSP               0
RREL               0
excess_ret         0
dtype: int64

In [211]:
data = data.iloc[1:].copy()

In [214]:
cay_series = data["CAY"].astype(float)
exog_vars = ["log_PD", "log_PE", "excess_ret"]
exog = data[exog_vars].astype(float)

kalman_model = sm.tsa.UnobservedComponents(
    endog=cay_series,
    level="local level",
    exog=exog
)

kalman_res = kalman_model.fit(disp=False)
cay_fitted = kalman_res.predict()

data["cay_imputed"] = cay_series.fillna(cay_fitted)
data["CAY"] = data["cay_imputed"]

print("# of NaN's after Imputation:", int(data["CAY"].isna().sum()))
print("\nTail:")
print(data[["CAY"]].tail(12))

# of NaN's after Imputation: 0

Tail:
                 CAY
2020-06-30 -0.039294
2020-07-31 -0.039836
2020-08-31 -0.041039
2020-09-30 -0.041449
2020-10-31 -0.041873
2020-11-30 -0.042195
2020-12-31 -0.043572
2021-01-31 -0.043469
2021-02-28 -0.043030
2021-03-31 -0.042450
2021-04-30 -0.042992
2021-05-31 -0.042775


/opt/anaconda3/envs/yfinance_env/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency ME will be used.
  self._init_dates(dates, freq)


In [215]:
X = data.drop(columns=["log_ret"]).shift(1)
y = data["log_ret"]

final_df = pd.concat([y, X], axis=1).dropna()

In [216]:
final_df.head(20)

,log_ret,monthly_return,IV,RV,VRP,log_PE,log_PD,CAY,BAA,AAA,GS10,TB3MS,DFSP,TMSP,RREL,excess_ret,cay_imputed
2008-03-31,-0.014079,-0.025783,0.065171,0.058995,0.006176,3.079178,3.875335,0.006081,6.82,5.53,3.74,2.12,1.29,1.62,-1.805000,-0.027531,0.006081
2008-04-30,0.048108,-0.013980,0.073852,0.075129,-0.001277,3.082243,3.840204,0.009843,6.89,5.51,3.51,1.26,1.38,2.25,-2.358333,-0.015024,0.009843
2008-05-31,0.013493,0.049283,0.046684,0.046326,0.000358,3.173155,3.875230,0.009843,6.97,5.55,3.68,1.29,1.42,2.39,-2.030000,0.048215,0.009843
2008-06-30,-0.091226,0.013585,0.033596,0.037002,-0.003406,3.250590,3.894051,0.009843,6.93,5.57,3.88,1.73,1.36,2.15,-1.340000,0.012155,0.009843
2008-07-31,-0.009495,-0.087189,0.049107,0.050400,-0.001293,3.262303,3.844112,-0.006757,7.07,5.68,4.10,1.86,1.39,2.24,-0.980833,-0.088725,-0.006757
2008-08-31,0.014559,-0.009450,0.059587,0.067019,-0.007431,3.233494,3.777876,-0.006757,7.16,5.67,4.01,1.63,1.49,2.38,-0.945000,-0.010798,-0.006757
2008-09-30,-0.102180,0.014665,0.042936,0.045502,-0.002566,3.289644,3.795272,-0.006757,7.15,5.64,3.89,1.72,1.51,2.17,-0.648333,0.013244,-0.006757
2008-10-31,-0.182786,-0.097133,0.095739,0.128689,-0.032950,3.276549,3.741993,0.019210,7.31,5.65,3.69,1.13,1.66,2.56,-1.008333,-0.098069,0.019210
2008-11-30,-0.073742,-0.167054,0.385188,0.243338,0.141850,3.303900,3.519277,0.019210,8.88,6.28,3.81,0.67,2.60,3.14,-1.199167,-0.167610,0.019210
2008-12-31,-0.005465,-0.071089,0.398750,0.159564,0.239186,3.555073,3.431947,0.019210,9.21,6.12,3.53,0.19,3.09,3.34,-1.422500,-0.071247,0.019210


In [218]:
# VIX3M laden
vix3m = yf.download("^VIX3M", start=START, end=END)

# "Close" robust extrahieren (bei yfinance ggf. MultiIndex-Spalten)
if isinstance(vix3m.columns, pd.MultiIndex):
    vix3m_close = vix3m["Close"].iloc[:, 0]
else:
    vix3m_close = vix3m["Close"]

# auf Monatsfrequenz bringen
vix3m_monthly = vix3m_close.resample("ME").mean().to_frame(name="VIX3M")

# mit bestehendem Datensatz mergen
data = data.join(vix3m_monthly, how="left")

# final_df neu bauen (konsistent mit deinem bisherigen Shift-Setup)
X = data.drop(columns=["log_ret"]).shift(1)
y = data["log_ret"]
final_df = pd.concat([y, X], axis=1).dropna()

final_df.tail()

[*********************100%***********************]  1 of 1 completed


,log_ret,monthly_return,IV,RV,VRP,log_PE,log_PD,CAY,BAA,AAA,GS10,TB3MS,DFSP,TMSP,RREL,excess_ret,cay_imputed,VIX3M
2021-01-31,-0.013283,0.027612,0.050212,0.031184,0.019028,3.670143,4.149581,-0.043572,3.16,2.26,0.93,0.09,0.90,0.84,-0.275000,0.027537,-0.043572,25.429091
2021-02-28,0.028761,-0.013195,0.063758,0.040120,0.023637,3.582525,4.179569,-0.043469,3.24,2.45,1.08,0.08,0.79,1.00,-0.165000,-0.013261,-0.043469,27.552632
2021-03-31,0.041269,0.029179,0.054342,0.037756,0.016587,3.503641,4.206646,-0.043030,3.42,2.70,1.26,0.04,0.72,1.22,-0.081667,0.029145,-0.043030,27.811579
2021-04-30,0.051760,0.042132,0.048388,0.050433,-0.002045,3.417831,4.217321,-0.042450,3.74,3.04,1.61,0.03,0.70,1.58,-0.070000,0.042107,-0.042450,25.341739
2021-05-31,0.004256,0.053123,0.030378,0.026464,0.003913,3.398683,4.273294,-0.042992,3.60,2.90,1.64,0.02,0.70,1.62,-0.070000,0.053106,-0.042992,21.223809


In [220]:
final_df.to_csv("data/final.csv")